# [기초-실습] 통계 101x데이터 분석: (5-6장) 가설검정

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## ⚙️ 환경 준비 — 한글 폰트 설치 및 라이브러리 불러오기

In [ ]:
# 구글 코랩 환경에서 한글 폰트 설치 및 설정하기
# 필요시 아래 코드 실행 후, [런타임] - [세션 다시 시작] 후 셀을 다시 실행하세요.
!pip install statsmodels
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

In [ ]:
# 파이썬 라이브러리 및 모듈 가져오기
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'NanumGothic'  # 기본 폰트 설정
plt.rcParams['axes.unicode_minus'] = False   # 마이너스 기호 깨짐 방지

# 문제 1. 게임 아이템 뽑기 확률, 정말 10%일까? (난이도: 하)

**📘 문제**

- 어떤 게임 개발사에서 새로운 아이템의 뽑기 확률이 **10%**로 설정되었다고 주장하고 있습니다.

- 그런데 유저 커뮤니티에서는 "실제 확률은 10%보다 낮은 것 같다"는 의혹이 제기되었습니다.

- 이를 확인하기 위해 한 유저가 아이템을 **200번 뽑았고, 그중 12번 성공**했습니다.

- 이번 실습에서는 **이항검정(Binomial Test)**으로 이 결과가 개발사의 주장을 기각할 만큼 유의미한지 확인하고,
  시뮬레이션을 통해 **p-값이 무엇을 의미하는지**를 직접 눈으로 확인해 봅니다.

In [ ]:
# 문제 설정
n = 200  # 총 시행 횟수
x = 12   # 관찰된 성공 횟수
p = 0.1  # 귀무가설에서의 성공 확률

print(f"관찰 결과: {n}번 시도 중 {x}번 성공 (관찰 성공률 {x/n:.1%})")

**📌 아래를 수행해 보세요:**

- 귀무가설(H₀)과 대립가설(H₁)을 세워 봅시다.

- 이항검정을 수행하여 p-값을 구하고, 유의수준 0.05를 기준으로 결론을 내려봅시다.

- **"만약 개발사의 주장대로 실제 확률이 정말 10%라면"** 200번 뽑기에서 성공 횟수가 어떻게 분포하는지 시뮬레이션해 봅시다.

- 히스토그램을 그리고, 우리가 관찰한 '12번 성공'이 얼마나 희귀한 일인지 관찰해 봅시다.

In [ ]:
# [문제 1] Q1. 귀무가설(H₀)과 대립가설(H₁)을 주석으로 작성해봅시다.
# H₀ (귀무가설): 뽑기 확률이 10퍼센트 이다.
# H₁ (대립가설): 뽑기 확률이 10퍼센트보다 낮다.

In [ ]:
# [문제 1] Q2. 이항검정을 수행하고 p-값을 계산해봅시다.
# '확률이 더 낮은 것 같다'는 주장을 검증하려면 alternative 인수를 어떻게 설정해야 할까요?

from scipy.stats import binomtest

# 이항검정
result = binomtest(
    k=12,              # 성공 횟수
    n=200,             # 전체 시행 횟수
    p=0.10,            # 귀무가설에서의 성공 확률
    alternative="less" # 실제 확률이 10%보다 낮은지 검정
)

print("p-value:", result.pvalue)

In [ ]:
# [문제 1] Q3. 유의수준 0.05를 기준으로 통계적 결론을 내리는 코드를 작성해봅시다.
# alpha = 0.05

alpha = 0.05

if result.pvalue < alpha:
    print("귀무가설을 기각합니다.")
    print("실제 아이템 뽑기 확률이 10%보다 낮다고 볼 통계적 근거가 있습니다.")
else:
    print("귀무가설을 기각하지 못합니다.")
    print("실제 아이템 뽑기 확률이 10%보다 낮다고 보기에는 근거가 부족합니다.")

In [ ]:
# [문제 1] Q4. H₀가 사실(실제 확률 = 10%)이라고 가정하고,
# 200번 뽑기를 10,000번 반복하는 시뮬레이션을 수행해봅시다.
# 힌트: numpy의 이항분포 랜덤 함수(np.random.binomial)를 사용해보세요.

import numpy as np

np.random.seed(42)  # 실행할 때마다 같은 결과가 나오도록 설정

simulations = np.random.binomial(
    n=200,      # 한 번의 실험에서 200번 뽑기
    p=0.10,     # 귀무가설이 주장하는 성공 확률 10%
    size=10000  # 실험을 10,000번 반복
)

print(simulations)
print("시뮬레이션 횟수:", len(simulations))

In [ ]:
# [문제 1] Q5. 시뮬레이션 결과를 히스토그램으로 그려봅시다.
# 실제 관찰값(12번 성공)을 세로선으로 함께 표시해 봅시다.

import matplotlib.pyplot as plt

plt.hist(simulations, bins=30, edgecolor="black")

plt.axvline(
    x=12,
    linestyle="--",
    label="실제 관찰값: 12번"
)

plt.xlabel("성공 횟수")
plt.ylabel("빈도")
plt.title("200번 뽑기 시뮬레이션 결과")
plt.legend()

plt.show()

**🧠 데이터를 어떻게 읽을까요?**

- Q2에서 구한 p-값은 시뮬레이션 그래프의 어느 부분에 해당하며, 무엇을 의미하나요?

- 시뮬레이션 그래프를 볼 때, 우리가 관찰한 '12회 성공'은 개발사의 주장이 맞다는 가정 하에 흔한 일인가요, 드문 일인가요?

- 이 시뮬레이션 경험을 통해 "p-값이 작으면 귀무가설을 기각한다"는 규칙을 친구에게 어떻게 더 쉽게 설명할 수 있을까요?

- 만약 200번이 아니라 1,000번을 뽑아 같은 성공률(6%)이 나왔다면, p-값은 어떻게 달라질까요?

In [ ]:
# [문제 1] 데이터를 어떻게 읽을까요?


## 🧠 데이터를 어떻게 읽을까요?

### 1. Q2에서 구한 p-값은 시뮬레이션 그래프의 어느 부분에 해당하며, 무엇을 의미하나요?

# p-값은 실제 관찰값인 **12번 성공 이하가 나온 왼쪽 꼬리 부분**에 해당한다.  
# 즉, 개발사의 주장대로 실제 성공 확률이 10%라고 가정했을 때, 12번 이하의 성공이 나올 확률을 의미한다.

### 2. 우리가 관찰한 12회 성공은 흔한 일인가요, 드문 일인가요?

# 12회 성공은 히스토그램의 왼쪽 꼬리 부분에 위치하므로 비교적 드문 결과이다.  
# 따라서 개발사의 주장대로 성공 확률이 10%라고 보기에는 다소 이상한 결과라고 볼 수 있다.

### 3. p-값이 작으면 귀무가설을 기각한다는 규칙을 쉽게 설명하면?

# 개발사의 주장이 맞다고 했을 때 지금과 같은 결과가 거의 나오지 않는다면, 개발사의 주장을 믿기 어렵다고 판단할 수 있다.  
# 그래서 p-값이 작으면 귀무가설을 기각한다.

### 4. 1,000번을 뽑아 같은 성공률 6%가 나왔다면 p-값은 어떻게 달라질까요?

# p-값은 지금보다 더 작아질 것이다.  
# 표본의 수가 많아질수록 10%와 6%의 차이를 단순한 우연으로 보기 어려워지기 때문에, 귀무가설을 기각할 근거가 더 강해진다.

# 문제 2. 과자 한 봉지의 중량은 150g이 맞을까? (난이도: 하)

**📘 문제**

- 한 식품 공장에서 생산하는 과자 한 봉지의 **목표 중량은 150g**입니다.

- 품질관리팀은 생산 공정이 목표 중량을 잘 맞추고 있는지 확인하기 위해,
  생산된 과자 **30봉지를 무작위로 추출**하여 무게를 측정했습니다.

- 이번 실습에서는 **일표본 t-검정(One-sample t-Test)**으로 표본 평균이 목표 중량과 유의미하게 다른지 검증하고,
  **"평균이 정확히 150g이라면"** 어떤 표본 평균들이 나올 수 있는지 시뮬레이션으로 확인해 봅니다.

In [ ]:
# 데이터 생성 (과자 30봉지의 무게 측정값)
np.random.seed(42)
sample_weights = np.random.normal(loc=151.5, scale=2, size=30)
pop_mean = 150  # 목표 중량 (모평균)

# 표본 분포 시각화
sns.histplot(sample_weights, bins=10, kde=True)
plt.axvline(x=pop_mean, color='gray', linestyle='--', label='목표 중량 150g')
plt.title("추출한 과자 30봉지의 무게 분포")
plt.xlabel("무게 (g)")
plt.legend()
plt.show()

**📌 아래를 수행해 보세요:**

- 귀무가설(H₀)과 대립가설(H₁)을 세워 봅시다.

- 일표본 t-검정을 수행하여 t-통계량과 p-값을 구하고, 유의수준 0.05를 기준으로 결론을 내려봅시다.

- **"평균 무게가 정확히 150g이라면"** 30개짜리 표본의 평균이 어떻게 분포하는지 10,000번 시뮬레이션해 봅시다.

- 히스토그램을 그리고, 실제 관찰된 표본 평균이 그 분포에서 어디에 위치하는지 관찰해 봅시다.

In [ ]:
# [문제 2] Q1. 귀무가설(H₀)과 대립가설(H₁)을 주석으로 작성해봅시다.
# H₀ (귀무가설): 과자 한 봉지의 평균 중량은 150g이다.
# H₁ (대립가설): 과자 한 봉지의 평균 중량은 150g이 아니다.

In [ ]:
# [문제 2] Q2. 일표본 t-검정을 수행하여 t-통계량과 p-값을 구해봅시다.

from scipy import stats

# 일표본 t-검정
t_stat, p_value = stats.ttest_1samp(sample_weights, pop_mean)

print("t-통계량:", t_stat)
print("p-값:", p_value)

In [ ]:
# [문제 2] Q3. 유의수준 0.05를 기준으로 통계적 결론을 내리는 코드를 작성해봅시다.
# alpha = 0.05

# 유의수준 설정
alpha = 0.05

# p-값과 유의수준 비교
if p_value < alpha:
    print("귀무가설을 기각합니다.")
    print("과자 한 봉지의 평균 중량은 150g과 유의미하게 다릅니다.")
else:
    print("귀무가설을 기각하지 못합니다.")
    print("과자 한 봉지의 평균 중량이 150g과 다르다고 볼 충분한 근거가 없습니다.")

In [ ]:
# [문제 2] Q4. H₀가 사실(평균 = 150g)이라고 가정하고, 30개짜리 표본을 10,000번 뽑아
# 각 표본의 평균을 리스트에 저장해봅시다.
# 힌트: 표준편차는 우리가 가진 표본의 표준편차를 사용한다고 가정하고, for 반복문을 사용해보세요.

sample_means = []

sample_std = sample_weights.std(ddof=1)

for _ in range(10000):
    sample = np.random.normal(loc=150, scale=sample_std, size=30)
    sample_means.append(sample.mean())

In [ ]:
# [문제 2] Q5. 시뮬레이션으로 얻은 표본 평균들의 분포를 히스토그램으로 그려봅시다.
# 실제 관찰된 표본 평균을 세로선으로 함께 표시해 봅시다.

# 실제 관찰된 표본 평균
observed_mean = sample_weights.mean()

# 시뮬레이션 표본 평균 분포 시각화
plt.hist(sample_means, bins=30)

# 실제 관찰된 표본 평균을 세로선으로 표시
plt.axvline(x=observed_mean, color="red", linestyle="--")

plt.xlabel("표본 평균")
plt.ylabel("빈도")
plt.title("H₀가 참일 때 표본 평균의 분포")
plt.show()

**🧠 데이터를 어떻게 읽을까요?**

- Q5의 히스토그램은 무엇을 나타내나요? 이 분포의 모양이 정규분포와 비슷한 이유는 무엇일까요? (힌트: 4장에서 배운 중심극한정리)

- 우리가 실제로 관찰한 표본 평균(세로선)은 이 분포에서 흔하게 나타나는 값인가요? Q2의 p-값과 이 시각적 위치는 어떻게 관련되나요?

- 만약 p-값이 0.001이었다면, 세로선은 그래프의 어디쯤에 위치할 것으로 예상되나요?

- 이 결과는 공장 입장에서 어떤 조치를 취해야 함을 시사할까요? (목표보다 무겁게 나오는 것은 어떤 손실일까요?)

In [ ]:
# [문제 2] 데이터를 어떻게 읽을까요?
#**1.**
#히스토그램은 **귀무가설(H₀)이 참일 때 나올 수 있는 표본평균의 분포**를 나타낸다. 중심극한정리 때문에 정규분포와 비슷한 모양이 된다.

#**2.**
#실제 표본평균은 분포의 꼬리 쪽에 있어 흔한 값이 아니다. 따라서 **p값이 작고 귀무가설을 기각**하게 된다.

#**3.**
#p값이 0.001이라면 세로선은 **그래프의 가장 끝 꼬리 부분**에 위치할 것이다.

#**4.**
#공장은 생산 공정을 점검해 평균 중량을 150g에 맞춰야 한다. 목표보다 무겁게 생산되면 **원재료 비용이 증가하는 손실**이 발생한다.
#

# 문제 3. 어떤 온라인 학습 방식이 더 효과적일까? (난이도: 중)

**📘 문제**

- 한 교육 기업에서 두 가지 다른 온라인 학습 방식(A, B)을 개발했습니다.

- 방식 A가 방식 B보다 학생들의 성적 향상에 더 효과적인지 알아보기 위해,
  두 그룹의 학생들에게 각각 다른 방식으로 한 달간 학습시킨 후 시험을 보게 했습니다.

- 이번 실습에서는 **이표본 t-검정(Two-sample t-Test)**으로 두 방식의 평균 점수 차이를 검증합니다.
  단, t-검정의 기본 가정인 **정규성**과 **등분산성**을 먼저 확인해야 합니다.

- 또한 **"두 방식의 효과가 완전히 똑같다면"** 우연히 나타날 수 있는 점수 차이의 범위를 시뮬레이션으로 확인해 봅니다.

In [ ]:
# 데이터 생성 (두 학습 방식의 시험 점수)
np.random.seed(0)
group_a_scores = np.random.normal(loc=85, scale=8, size=50)
group_b_scores = np.random.normal(loc=80, scale=7, size=50)

df_scores = pd.DataFrame({'점수': np.concatenate([group_a_scores, group_b_scores]),
                          '학습방식': ['A']*50 + ['B']*50})

# 두 그룹의 점수 분포 시각화
sns.boxplot(x='학습방식', y='점수', data=df_scores)
plt.title("학습 방식에 따른 시험 점수 분포")
plt.show()

**📌 아래를 수행해 보세요:**

- 두 그룹에 대해 **정규성 검정(Shapiro-Wilk test)**을 수행해 봅시다. (H₀: 데이터는 정규분포를 따른다.)

- **등분산성 검정(Levene's test)**을 수행해 봅시다. (H₀: 두 그룹의 분산은 같다.)

- 가정 검토 결과에 따라 `equal_var` 인수를 설정하여 **이표본 t-검정**을 수행해 봅시다.

- 두 그룹 데이터를 모두 합친 뒤 무작위로 다시 나누는 방식으로, **"두 방식의 효과가 같을 때"**의 평균 차이 분포를 시뮬레이션해 봅시다.

In [ ]:
# [문제 3] Q1. 두 그룹에 대해 각각 정규성 검정(Shapiro-Wilk test)을 수행하고 p-값을 출력해봅시다.
# H₀: 데이터는 정규분포를 따른다.

from scipy.stats import shapiro

# A 그룹 정규성 검정
stat_a, p_a = shapiro(group_a_scores)

# B 그룹 정규성 검정
stat_b, p_b = shapiro(group_b_scores)

print("A 그룹 p-value:", p_a)
print("B 그룹 p-value:", p_b)

In [ ]:
# [문제 3] Q2. 두 그룹에 대해 등분산성 검정(Levene's test)을 수행하고 p-값을 출력해봅시다.
# H₀: 두 그룹의 분산은 같다.

from scipy.stats import levene

stat, p = levene(group_a_scores, group_b_scores)

print("Levene's test p-value:", p)

In [ ]:
# [문제 3] Q3. 이표본 t-검정을 수행하여 t-통계량과 p-값을 구해봅시다.
# H₀: 두 그룹의 평균은 같다. / H₁: 두 그룹의 평균은 다르다.
# 등분산성 가정을 만족했는지 여부에 따라 equal_var 인수를 설정해보세요.

from scipy.stats import ttest_ind

t_stat, p_value = ttest_ind(
    group_a_scores,
    group_b_scores,
    equal_var=True
)

print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
# [문제 3] Q4. H₀가 사실(두 그룹의 평균이 같다)이라고 가정하고 시뮬레이션을 10,000번 수행해봅시다.
# 두 그룹의 데이터를 모두 합친 뒤(np.concatenate), 매 반복마다 비복원추출로 50개(가상 A그룹)와
# 나머지 50개(가상 B그룹)로 나누어 두 그룹의 평균 차이를 계산하고 리스트에 저장해보세요.
# 힌트: np.random.permutation 을 사용해보세요.

# 두 그룹 데이터를 하나로 합치기
combined_scores = np.concatenate([group_a_scores, group_b_scores])

# 시뮬레이션에서 나온 평균 차이를 저장할 리스트
simulated_mean_diffs = []

# 10,000번 반복
for _ in range(10000):
    # 100명의 점수를 무작위 순서로 섞기
    shuffled_scores = np.random.permutation(combined_scores)

    # 앞의 50개는 가상 A그룹
    simulated_a = shuffled_scores[:50]

    # 나머지 50개는 가상 B그룹
    simulated_b = shuffled_scores[50:]

    # 가상 A와 B의 평균 차이를 계산해서 저장
    mean_diff = np.mean(simulated_a) - np.mean(simulated_b)
    simulated_mean_diffs.append(mean_diff)

print("시뮬레이션 횟수:", len(simulated_mean_diffs))
print("평균 차이 예시:", simulated_mean_diffs[:5])

In [ ]:
# [문제 3] Q5. 시뮬레이션으로 얻은 평균 차이의 분포를 히스토그램으로 그려봅시다.
# 실제 관찰된 평균 차이(A그룹 평균 - B그룹 평균)를 세로선으로 함께 표시해 봅시다.

# 실제 평균 차이
observed_diff = np.mean(group_a_scores) - np.mean(group_b_scores)

# 히스토그램
plt.figure(figsize=(8, 5))
plt.hist(simulated_mean_diffs, bins=30, edgecolor='black', alpha=0.7)

# 실제 평균 차이 표시
plt.axvline(observed_diff, color='red', linestyle='--', linewidth=2,
            label='Observed Mean Difference')

plt.xlabel("Mean Difference (A - B)")
plt.ylabel("Frequency")
plt.title("Simulation Distribution under H₀")
plt.legend()

plt.show()

**🧠 데이터를 어떻게 읽을까요?**

- 정규성 검정과 등분산성 검정의 p-값은 각각 어떻게 해석해야 하나요? 이 결과는 이표본 t-검정을 사용하는 데 문제가 없음을 보여주나요?

- Q5의 히스토그램 중심이 0에 가까운 이유는 무엇일까요?

- 실제 관찰된 평균 차이(세로선)는 "두 방식의 효과가 같다"고 가정했을 때 우연히 나타날 수 있는 범위 안에 있나요, 벗어나나요?

- 이 시각적 판단과 Q3의 p-값은 어떤 관계가 있나요?

- 통계적으로 유의미한 차이가 있다면, 교육 기업은 이 결과를 어떻게 활용할 수 있을까요?

In [ ]:
### 데이터를 어떻게 읽을까요?

# - **정규성 검정**과 **등분산성 검정**의 p값이 모두 0.05보다 크므로 정규성과 등분산성을 만족하며, 이표본 t-검정을 사용하는 데 문제가 없습니다.

# - Q5의 히스토그램 중심이 0에 가까운 이유는 귀무가설(H₀: 두 그룹의 평균이 같다)을 가정했기 때문입니다.

# - 실제 평균 차이(세로선)는 히스토그램의 꼬리 쪽에 위치하여 우연히 나타날 가능성이 낮습니다.

# - 세로선이 꼬리 쪽에 있을수록 p값은 작아지며, 귀무가설을 기각할 가능성이 커집니다.

# - 통계적으로 유의미한 차이가 있다면, 교육 기업은 평균 점수가 더 높은 학습 방식을 교육 프로그램에 활용할 수 있습니다.

# 문제 4. 연령대별로 선호하는 영화 장르가 다를까? (난이도: 중)

**📘 문제**

- 한 영화관에서 고객의 **연령대(20대, 30대, 40대)**에 따라 선호하는 **영화 장르(액션, 로맨스)**에 차이가 있는지 궁금해졌습니다.

- 이를 알아보기 위해 300명의 고객을 대상으로 설문조사를 진행하여 분할표를 얻었습니다.

- 이번 실습에서는 **카이제곱 독립성 검정(Chi-squared Test of Independence)**으로
  두 범주형 변수 사이에 연관성이 있는지 검증합니다.

- 또한 **"두 변수가 서로 아무 관련이 없다면(독립이라면)"** 어떤 카이제곱 통계량들이 나올 수 있는지 시뮬레이션으로 확인해 봅니다.

In [ ]:
# 데이터 생성 (관측 빈도 분할표)
data = {'액션': [70, 50, 30],
        '로맨스': [30, 60, 60]}
observed = pd.DataFrame(data, index=['20대', '30대', '40대'])

print("관측 빈도 (Observed Frequencies):")
print(observed)

# 분할표 시각화
observed.plot(kind='bar', figsize=(8, 5))
plt.title("연령대별 선호 장르 응답 수")
plt.xlabel("연령대")
plt.ylabel("응답 수 (명)")
plt.xticks(rotation=0)
plt.show()

**📌 아래를 수행해 보세요:**

- 귀무가설(H₀)과 대립가설(H₁)을 세워 봅시다.

- 카이제곱 독립성 검정을 수행하여 카이제곱 통계량, p-값, 자유도, **기대 빈도**를 구해봅시다.

- **"연령대와 선호 장르가 독립이라면"** 어떤 분할표들이 나타날 수 있는지 5,000번 시뮬레이션해 봅시다.

- 시뮬레이션으로 얻은 카이제곱 통계량 분포와 실제 관찰된 카이제곱 통계량을 비교해 봅시다.

In [ ]:
# [문제 4] Q1. 귀무가설(H₀)과 대립가설(H₁)을 주석으로 작성해봅시다.
# H₀ (귀무가설): 연령대와 선호하는 영화 장르는 서로 관련이 없다.
# H₁ (대립가설): 연령대와 선호하는 영화 장르는 서로 관련이 있다.

In [ ]:
# [문제 4] Q2. 카이제곱 독립성 검정을 수행하고 카이제곱 통계량, p-값, 자유도, 기대 빈도를 구해봅시다.
# 기대 빈도는 관측 빈도와 비교하기 쉽도록 데이터프레임 형태로 출력해보세요.

from scipy.stats import chi2_contingency

# 카이제곱 독립성 검정
chi2, p_value, dof, expected = chi2_contingency(observed)

# 기대 빈도를 DataFrame으로 변환
expected_df = pd.DataFrame(
    expected,
    index=observed.index,
    columns=observed.columns
)

print("카이제곱 통계량:", chi2)
print("p-value:", p_value)
print("자유도:", dof)

print("\n기대 빈도 (Expected Frequencies):")
print(expected_df)

In [ ]:
# [문제 4] Q3. H₀가 사실(두 변수가 독립)이라고 가정하고 시뮬레이션을 5,000번 수행해봅시다.
# 각 반복마다 300명의 가상 고객에게 연령대와 장르를 '독립적으로' 할당한 후,
# 가상 분할표를 만들고 카이제곱 통계량을 계산하여 리스트에 저장해보세요.
# 힌트 1: 전체 비율(observed.sum(axis=0) / 전체, observed.sum(axis=1) / 전체)을 확률로 사용
# 힌트 2: np.random.choice 로 300명의 연령대와 장르를 각각 독립적으로 생성
# 힌트 3: pd.crosstab 으로 분할표 생성 후 stats.chi2_contingency 로 통계량 계산

from scipy import stats

# 전체 고객 수
n = observed.values.sum()

# 실제 데이터의 연령대 비율과 장르 비율
age_probs = observed.sum(axis=1) / n
genre_probs = observed.sum(axis=0) / n

simulated_chi2 = []

for _ in range(5000):
    # H₀처럼 연령대와 장르를 서로 독립적으로 배정
    simulated_ages = np.random.choice(
        observed.index,
        size=n,
        p=age_probs
    )

    simulated_genres = np.random.choice(
        observed.columns,
        size=n,
        p=genre_probs
    )

    # 가상 분할표 생성
    simulated_table = pd.crosstab(
        simulated_ages,
        simulated_genres
    )

    # 행과 열 순서를 원래 분할표와 동일하게 정리
    simulated_table = simulated_table.reindex(
        index=observed.index,
        columns=observed.columns,
        fill_value=0
    )

    # 카이제곱 통계량 계산
    chi2_sim, _, _, _ = stats.chi2_contingency(simulated_table)

    simulated_chi2.append(chi2_sim)

print("시뮬레이션 횟수:", len(simulated_chi2))
print("처음 5개의 카이제곱 통계량:", simulated_chi2[:5])

In [ ]:
# [문제 4] Q4. 시뮬레이션으로 얻은 카이제곱 통계량의 분포를 히스토그램으로 그려봅시다.
# 실제 관찰된 카이제곱 통계량을 세로선으로 함께 표시해 봅시다.

plt.figure(figsize=(8, 5))

plt.hist(simulated_chi2, bins=30, edgecolor='black', alpha=0.7)

# 실제 카이제곱 통계량
plt.axvline(
    chi2,
    color='red',
    linestyle='--',
    linewidth=2,
    label=f'Observed Chi-square = {chi2:.2f}'
)

plt.title("Simulated Chi-square Distribution under H₀")
plt.xlabel("Chi-square Statistic")
plt.ylabel("Frequency")
plt.legend()

plt.show()

**🧠 데이터를 어떻게 읽을까요?**

- '기대 빈도'는 어떤 의미를 가지며, '관측 빈도'와의 차이가 클수록 카이제곱 통계량은 어떻게 변할까요?

- Q4의 히스토그램은 어떤 분포를 시각화한 것인가요? 실제 관찰된 카이제곱 통계량(세로선)은 흔한 값인가요, 극단적인 값인가요?

- 유의수준 0.05에서, 연령대와 선호 장르 사이에 통계적으로 유의미한 연관성이 있다고 결론 내릴 수 있나요? 그 이유는 무엇인가요?

- 이 결과를 바탕으로 영화관은 연령대별로 어떤 상영 전략이나 프로모션을 세울 수 있을까요?

In [ ]:
# [문제 4] 데이터를 어떻게 읽을까요?
# - 기대 빈도는 연령대와 선호 장르가 서로 관련이 없다고 가정했을 때 예상되는 인원수입니다. 관측 빈도와 기대 빈도의 차이가 클수록 카이제곱 통계량은 커집니다.

# - Q4의 히스토그램은 귀무가설이 맞을 때 나올 수 있는 카이제곱 통계량의 분포입니다. 실제 카이제곱 통계량은 약 26.91로 분포의 오른쪽 꼬리에 위치하므로 매우 극단적인 값입니다.

# - p값이 0.05보다 매우 작으므로 귀무가설을 기각합니다. 따라서 연령대와 선호 장르 사이에는 통계적으로 유의미한 연관성이 있다고 볼 수 있습니다.

# - 영화관은 20대에게는 액션 영화 할인이나 관련 굿즈를 제공하고, 30대와 40대에게는 로맨스 영화 중심의 상영 시간표와 맞춤형 프로모션을 제공할 수 있습니다.

# 문제 5. 미니 프로젝트 - 어떤 신규 비료가 가장 효과적일까? (난이도: 상)

**📘 문제**

- 한 농업 연구소에서 새로 개발한 비료 3종류(A, B, C)의 생산량 증대 효과를 비교하고자 합니다.

- 동일한 조건의 밭 30개를 준비하여, 각각 10개씩 비료 A, B, C를 투여한 후 수확량을 측정했습니다.

- 이번 실습에서는 **분산분석(ANOVA)**으로 세 비료 간 평균 수확량에 차이가 있는지 검증하고,
  차이가 있다면 **사후분석(Tukey's HSD)**으로 어떤 비료끼리 차이가 나는지 확인합니다.

- 마지막으로 **"세 비료의 효과가 완전히 똑같다면"** 어떤 F-통계량이 나올 수 있는지 시뮬레이션하고,
  분석 결과를 바탕으로 연구소에 어떤 비료를 추천할지까지 생각해 봅니다.

In [ ]:
# 데이터 생성 (비료 종류별 수확량)
np.random.seed(123)
fertilizer_a = np.random.normal(loc=105, scale=10, size=10)
fertilizer_b = np.random.normal(loc=120, scale=12, size=10)
fertilizer_c = np.random.normal(loc=108, scale=9, size=10)

# 데이터프레임으로 변환 (사후분석을 위해)
df = pd.DataFrame({'수확량': np.concatenate([fertilizer_a, fertilizer_b, fertilizer_c]),
                   '비료종류': ['A']*10 + ['B']*10 + ['C']*10})

# 데이터 탐색 시각화
sns.boxplot(x='비료종류', y='수확량', data=df)
plt.title("비료 종류에 따른 수확량 분포")
plt.show()

**📌 아래를 수행해 보세요:**

- 세 그룹에 대해 분산분석(ANOVA)을 수행하여 F-통계량과 p-값을 구해봅시다.

- ANOVA 결과가 유의미할 경우에만 **Tukey's HSD 사후분석**을 수행하고 결과를 해석해 봅시다.

- **"세 비료의 효과가 완전히 똑같다면"** 어떤 F-통계량들이 나오는지 5,000번 시뮬레이션해 봅시다.

- 히스토그램을 그리고, 실제 관찰된 F-통계량이 얼마나 극단적인 값인지 확인해 봅시다.

- 분석 결과를 종합하여 연구소에 추천할 비료를 정해 봅시다.

In [ ]:
# [문제 5] Q1. 분산분석(ANOVA)을 수행하여 F-통계량과 p-값을 구해봅시다.
# H₀: 세 비료의 평균 수확량은 모두 같다.
# H₁: 적어도 하나 이상의 비료 평균 수확량은 다르다.

from scipy import stats

# 일원분산분석(One-way ANOVA)
f_stat, p_value = stats.f_oneway(
    fertilizer_a,
    fertilizer_b,
    fertilizer_c
)

print("F-통계량:", f_stat)
print("p-value:", p_value)

In [ ]:
# [문제 5] Q2. ANOVA의 p-값이 유의수준 0.05보다 작을 경우에만
# Tukey's HSD 사후분석을 수행하고 결과를 출력해봅시다.
# 힌트: pairwise_tukeyhsd(endog=수확량, groups=비료종류, alpha=0.05)

from statsmodels.stats.multicomp import pairwise_tukeyhsd

if p_value < 0.05:
    tukey_result = pairwise_tukeyhsd(
        endog=df['수확량'],
        groups=df['비료종류'],
        alpha=0.05
    )

    print(tukey_result)

else:
    print("ANOVA 결과가 유의하지 않아 사후분석을 수행하지 않습니다.")

In [ ]:
# [문제 5] Q3. H₀가 사실(세 그룹의 평균이 모두 같다)이라고 가정하고 시뮬레이션을 5,000번 수행해봅시다.
# 모든 데이터를 합친 평균과 표준편차를 가진 H₀ 모집단에서 10개씩 3개의 가상 그룹을 생성하고,
# 이 가상 그룹들로 ANOVA를 수행하여 F-통계량을 리스트에 저장해보세요.

# 전체 데이터를 하나로 합치기
all_yields = np.concatenate([
    fertilizer_a,
    fertilizer_b,
    fertilizer_c
])

# H₀ 모집단의 평균과 표준편차
null_mean = np.mean(all_yields)
null_std = np.std(all_yields, ddof=1)

# 시뮬레이션으로 얻은 F-통계량 저장
simulated_f_stats = []

for _ in range(5000):
    # 세 비료의 평균이 같다고 가정한 모집단에서 각각 10개씩 생성
    simulated_a = np.random.normal(
        loc=null_mean,
        scale=null_std,
        size=10
    )

    simulated_b = np.random.normal(
        loc=null_mean,
        scale=null_std,
        size=10
    )

    simulated_c = np.random.normal(
        loc=null_mean,
        scale=null_std,
        size=10
    )

    # 가상 그룹으로 ANOVA 수행
    simulated_f, _ = stats.f_oneway(
        simulated_a,
        simulated_b,
        simulated_c
    )

    # F-통계량 저장
    simulated_f_stats.append(simulated_f)

print("시뮬레이션 횟수:", len(simulated_f_stats))
print("시뮬레이션 F-통계량 일부:", simulated_f_stats[:5])

In [ ]:
# [문제 5] Q4. 시뮬레이션으로 얻은 F-통계량의 분포를 히스토그램으로 그려봅시다.
# 실제 관찰된 F-통계량을 세로선으로 함께 표시해 봅시다.

plt.figure(figsize=(8, 5))

# 시뮬레이션으로 얻은 F-통계량 분포
plt.hist(
    simulated_f_stats,
    bins=30,
    edgecolor='black',
    alpha=0.7
)

# 실제 F-통계량 표시
plt.axvline(
    f_stat,
    color='red',
    linestyle='--',
    linewidth=2,
    label=f'Observed F = {f_stat:.2f}'
)

plt.title("Simulated F Distribution under H₀")
plt.xlabel("F Statistic")
plt.ylabel("Frequency")
plt.legend()

plt.show()

In [ ]:
# [문제 5] Q5. 분석 결과를 바탕으로 연구소에 어떤 비료를 추천하시겠습니까?
### 연구소에 대한 비료 추천

# ANOVA 결과 p값이 0.05보다 작아 세 비료의 평균 수확량에는 통계적으로 유의미한 차이가 있다고 판단했습니다.

# 평균 수확량은 비료 B가 약 125.98로 가장 높았으며, Tukey 사후분석에서도 비료 B는 A와 C보다 유의하게 높은 수확량을 보였습니다. 반면 비료 A와 C 사이에는 유의미한 차이가 없었습니다.

# 따라서 생산량을 높이는 것이 목적이라면 비료 B를 추천합니다. 다만 이번 실험은 비료별로 10개의 밭만 사용했기 때문에, 실제 도입 전에는 더 많은 밭과 다양한 환경에서 추가 실험을 진행하는 것이 좋습니다.

**🧠 데이터를 어떻게 읽을까요?**

- ANOVA 검정의 p-값을 통해 어떤 결론을 내릴 수 있나요? 이 결과는 "세 비료가 모두 동일한 효과를 가진다"는 뜻인가요, "적어도 하나는 다르다"는 뜻인가요?

- Tukey's HSD 결과표의 `reject` 열에서 `True`로 표시된 조합은 무엇이며, 이는 무엇을 의미하나요?

- `meandiff` 열의 값은 어떤 정보를 주나요? 신뢰구간(`lower`, `upper`)이 0을 포함하는지 여부는 무엇을 뜻할까요?

- 그룹별 표본 크기가 10개로 작다는 점은 이 결론의 신뢰도에 어떤 영향을 줄까요? 어떻게 개선할 수 있을까요?

In [ ]:
# [문제 5] 데이터를 어떻게 읽을까요?

# ▸ **ANOVA 검정의 p값으로 내릴 수 있는 결론**

# ANOVA의 p값이 0.05보다 작으므로 귀무가설(H₀)을 기각합니다. 따라서 **세 비료의 평균 수확량이 모두 같다고 볼 수 없으며, 적어도 하나 이상의 비료는 다른 비료와 평균 수확량에 차이가 있다고 해석합니다.** 하지만 ANOVA만으로는 어떤 비료끼리 차이가 있는지는 알 수 없습니다.


# ▸ **Tukey's HSD의 reject 열 의미**

# reject가 **True**인 조합은 두 비료의 평균 차이가 통계적으로 유의하다는 뜻입니다. 이번 결과에서는 **A-B와 B-C는 True**, **A-C는 False**로 나타났습니다. 즉, 비료 B는 A와 C보다 수확량이 유의하게 높았고, A와 C는 통계적으로 유의한 차이가 없었습니다.


# ▸ **meandiff와 신뢰구간(lower, upper)의 의미**

# meandiff는 두 비료의 평균 수확량 차이를 의미합니다. 값이 클수록 평균 차이가 크다는 뜻입니다.

# 신뢰구간(lower, upper)이 **0을 포함하지 않으면 두 평균의 차이가 유의하다**고 판단하며, **0을 포함하면 평균 차이가 있다고 보기 어렵습니다.** 따라서 reject 결과와 같은 결론을 얻게 됩니다.


# ▸ **표본 크기가 10개인 영향과 개선 방법**

# 각 그룹의 표본이 10개로 적기 때문에 우연에 의해 결과가 영향을 받을 가능성이 있습니다. 또한 모집단을 충분히 대표하지 못할 수도 있어 결과의 신뢰도가 다소 낮아질 수 있습니다.

# 이를 개선하려면 **각 비료의 표본 수를 더 늘리고**, 다양한 토양이나 재배 환경에서 반복 실험을 수행하면 더 신뢰할 수 있는 결과를 얻을 수 있습니다.